# Experimental data: radial k-space

Every earlier notebook samples a function it defines itself. Measured data arrives in the layout of the instrument that recorded it. Radial MRI records k-space, the 2-D Fourier transform of the image, along *lines* through the center of k-space, each at its own angle, with equally spaced readout samples on both sides of the center. PyPFT stores polar data as *spokes* instead: half-lines starting at the center, in the `(radial, angular[, batch])` layout with a centered angular axis. `pypft.acquisition` converts the first into the second.

This notebook uses a public radial cine cardiac MRI dataset [[1]](#ref-1), acquired for a reconstruction study [[2]](#ref-2). A small excerpt of one of its files is part of the repository: one cardiac phase of one subject, its four strongest receiver coils. The notebook converts lines into spokes, reads the file as a single sample and as a batch, reconstructs an image along both of PyPFT's paths, and ends with converting data from any other source.

In [ ]:
# Draw every figure below the cell that makes it
%matplotlib inline

# Paths and temporary directories, NumPy for arrays, matplotlib for figures
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# PyPFT, and its acquisition package
import pypft
from pypft.acquisition import (
    RadialLineLayout,
    NonUniformAnglesError,
    RadialKSpace,
    holds_several_arrays,
    load_stored_array,
    radial_lines_to_polar,
    select_radial_kspace,
    selectable_axes,
    stored_arrays,
)

## From lines to spokes

A line at angle $\varphi$ passes through the center, so it holds two spokes: its samples after the center lie on the spoke at $\varphi$, and its samples before the center, read backward, lie on the spoke at $\varphi + \pi$. Both start at the center sample. With $L$ lines at the uniform angles $m \pi / L$, $m = 0, \dots, L - 1$, the $2L$ spokes are exactly the uniform, centered spokes of a polar array with $2L$ angular samples: spoke $i$ at angle $(i - L)\pi / L$. The figure shows six lines and the twelve spokes they become, in PyPFT's angle convention (measured on image coordinates, from $+x$ toward $+y$, so the $y$ axis points down).

In [ ]:
# Six lines at uniform angles over half a turn
n_lines = 6
line_angles = np.arange(n_lines) * (np.pi / n_lines)
colors = plt.get_cmap(name="tab10")(np.arange(n_lines))

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
for line, (angle, color) in enumerate(zip(line_angles, colors)):
    direction = np.array(object=[np.cos(angle), np.sin(angle)])
    # Left: the whole line, its arrow pointing along the samples after the center
    axes[0].annotate(
        text="",
        xy=direction,
        xytext=-direction,
        arrowprops={"arrowstyle": "->", "color": color, "linewidth": 2},
    )
    axes[0].annotate(text=f"line {line}", xy=1.12 * direction, color=color, ha="center")
    # Right: the two spokes of the line, labeled with their centered angular index
    for spoke_angle in (angle, angle + np.pi):
        spoke_direction = np.array(object=[np.cos(spoke_angle), np.sin(spoke_angle)])
        index = int(round(spoke_angle / (np.pi / n_lines))) % (2 * n_lines)
        centered = (index + n_lines) % (2 * n_lines)
        axes[1].annotate(
            text="",
            xy=spoke_direction,
            xytext=(0, 0),
            arrowprops={"arrowstyle": "->", "color": color, "linewidth": 2},
        )
        axes[1].annotate(
            text=str(centered), xy=1.12 * spoke_direction, color=color, ha="center"
        )

# Image coordinates: y points down, as in PyPFT's angle convention
for ax, title in zip(axes, ("6 lines through the center", "12 spokes, centered index")):
    ax.set_xlim(left=-1.3, right=1.3)
    ax.set_ylim(bottom=1.3, top=-1.3)
    ax.set_aspect(aspect="equal")
    ax.set_xlabel(xlabel="$k_x$")
    ax.set_ylabel(ylabel="$k_y$")
    ax.set_title(label=title)
plt.show()

Spoke $6$, the centered index of angle $0$, holds the samples after the center of line $0$; spoke $0$, at angle $-\pi$, holds the samples before it. The order of the lines in the file does not matter, only their angles: sequential, interleaved and reversed orderings all fill the same twelve spokes. Golden-angle trajectories, whose angles are not multiples of $\pi / L$, have no uniform polar layout and are rejected (see the last section).

## Reading a file

Reading has two independent parts: the file, and the layout of the array in it. `load_stored_array` reads an array from a file, whatever its layout: a `.npy` file holds a single array, read as it is, while a `.npz` or `.mat` file holds named arrays, which `stored_arrays` lists with their shapes (from the file's headers alone) and one of which is read by name; `holds_several_arrays` tells the two kinds apart. The dataset's files are MATLAB files with one array, `data`, of shape `(slices, phases, readout, lines, coils, 2)`, the last axis holding the real and imaginary parts. `pypft.acquisition.RadialLineLayout` names the array layouts of radial lines PyPFT reads, by their axes in order, whether the lines were measured or generated, and whatever file they were stored in; this one is `SLICE_PHASE_READOUT_LINE_COIL_COMPLEX`. `selectable_axes` lists every axis of the array besides the readout and the lines: the axes a selection chooses from. The committed excerpt keeps the dataset's layout, with one slice, one phase, the 416 readout samples around the center of k-space and four coils. A full file of the dataset has 25 phases and 15 to 22 coils; to use one, download it from the dataset's page and change `path` to it.

In [ ]:
# The excerpt of the dataset that pypft.samples downloads into the test samples; change this to a full dataset file to use it
path = Path("../tests/samples/P01_slc1_phs1_lins196_cols416_cha4.mat")
# A MATLAB file holds named arrays: list them, then read the one holding the lines
print(f"several arrays: {holds_several_arrays(path=path)}")
print(stored_arrays(path=path))
lines = load_stored_array(path=path, name="data")

# How the array is laid out: its axes, in order
layout = RadialLineLayout.SLICE_PHASE_READOUT_LINE_COIL_COMPLEX

# The axes a selection chooses from, with their sizes
axes_sizes = selectable_axes(values=lines, layout=layout)
print(axes_sizes)

## A single sample or a batch

`select_radial_kspace` takes a *selection*: for each selectable axis, either one index, which picks that index and drops the axis, or `None`, which keeps the whole axis. An axis the selection leaves out takes index `0`. With every axis at one index the result is a single sample `(radial, angular)`; every axis kept goes into PyPFT's one batch axis, `(radial, angular, batch)`. Several kept axes are flattened into it in the array's order, the last one varying fastest, so a `reshape` separates them again.

The result is a `RadialKSpace`: its `values`, and the frequency `radius` they cover. Radial sample $k$ of $n$ sits at frequency $k \cdot \text{radius} / n$ on every spoke, in radians per pixel, where a pixel is one sample of the image an inverse DFT of one readout gives. Here the 416-sample readout, centered at sample 208, keeps 208 samples per spoke, up to the readout's own Nyquist frequency $\pi$.

In [ ]:
# One coil: a single sample
single = select_radial_kspace(values=lines, layout=layout, selection={"coil": 0})
print(f"single sample: {single.values.shape}, radius {single.radius:.4f} rad/pixel")

# Every coil: a batch, one sample per coil
batch = select_radial_kspace(values=lines, layout=layout, selection={"coil": None})
print(f"batch of coils: {batch.values.shape}")

# The batch's first sample is the single sample
print(f"same data: {np.array_equal(a1=batch.values[..., 0], a2=single.values)}")

The converted data is a uniform polar array in the frequency domain. Its magnitude, on a logarithmic scale, shows the center of k-space at radial sample 0 on every spoke. It also shows an asymmetric echo: the scanner recorded only 120 samples before the center of each line, so the spokes from the first half of the angles, which hold the samples before the center, are zero beyond radial sample 120.

In [ ]:
# Log-magnitude of the first coil's spokes, with the uniform frequencies and angles
magnitude = np.log10(np.abs(single.values) + 1e-12)
n_radial, n_angular = single.values.shape
frequency_extent = single.radius * (n_radial - 1) / n_radial

fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(
    X=magnitude,
    aspect="auto",
    origin="lower",
    extent=(-np.pi, np.pi * (1 - 2 / n_angular), 0.0, frequency_extent),
    vmin=magnitude.max() - 5,
)
ax.set_xlabel(xlabel="spoke angle $\\psi$ (rad)")
ax.set_ylabel(ylabel="frequency $\\rho$ (rad/pixel)")
ax.set_title(label="Coil 0, uniform polar k-space")
fig.colorbar(mappable=image, ax=ax, label="$\\log_{10} |F|$")
plt.show()

## Choosing a grid

The transform needs a `PolarGrid` in the same units. The image the readout's inverse DFT gives spans 416 pixels, so a space limit $R = 208$ pixels covers everything the acquisition can see. The grid's frequency samples must lie inside the data, $\max \rho \le$ `radius`, since the grid cannot use samples the data does not have; the high-order spokes reach farthest. The data has 392 spokes; a grid with 98 of them keeps the computation short and resolves the heart, at the cost of some angular detail.

In [ ]:
# A space-limited grid over the readout's field of view, in pixels
grid = pypft.PolarGrid(n_radial=128, n_angular=98, R=208.0)

# Its frequency samples must stay inside the data's radius
print(f"largest grid frequency {grid.rho.max():.3f} <= data radius {batch.radius:.3f}")

## The exact path

PyPFT's default path, the one every earlier notebook uses, needs samples on the grid's own frequency points. `pypft.resample_uniform_polar_frequency` interpolates the uniform data along each spoke onto `grid.rho`, the frequency-domain counterpart of `pypft.resample_uniform_polar` from [02_sampling_grids](02_sampling_grids.ipynb). `pypft.inverse_pft` then transforms every coil of the batch at once, and the root of the sum of squares over the coils combines them into one image.

In [ ]:
# Uniform k-space onto the grid's frequency points, every coil at once
F = pypft.resample_uniform_polar_frequency(
    values=batch.values, grid=grid, radius=batch.radius
)
print(f"on the grid: {F.shape}")

# The exact inverse, then the coils combined by their root sum of squares
f_exact = pypft.inverse_pft(F=F, grid=grid)
image_exact = np.sqrt(np.sum(a=np.abs(f_exact) ** 2, axis=-1))

## The ring-consistent route

The exact path identifies every spoke's angular index with a harmonic order, on both sides of the transform. [05_pft_and_ipft](05_pft_and_ipft.ipynb) measures what that costs: little for content near the center of a small grid, a lot for content away from the center on a grid with many spokes. Measured k-space is the second case: a body fills much of the field of view, and the data has hundreds of spokes. The ring-consistent route avoids the identification, and uniform polar data is its natural input: equal radii on every spoke make each radial index a true ring, so `pypft.inverse_pft_ring` with `radius` interpolates only along the radius, onto every harmonic's own frequencies, and takes the angular sums over the data's own 392 spokes. It is not exactly invertible, which does not matter here: the data is the measurement, and the image is the result.

In [ ]:
# The ring-consistent inverse, straight from the uniform data, every coil at once
f_ring = pypft.inverse_pft_ring(F=batch.values, grid=grid, radius=batch.radius)
image_ring = np.sqrt(np.sum(a=np.abs(f_ring) ** 2, axis=-1))

# Both images on a Cartesian pixel grid, for display only
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
for ax, image, title in zip(
    axes, (image_exact, image_ring), ("Exact path", "Ring-consistent route")
):
    signal = pypft.PolarSpatialAngularSignal(values=image, grid=grid)
    pypft.render_cartesian(signal=signal, height=300, width=300, ax=ax)
    ax.set_title(label=title)
plt.show()

The ring-consistent route shows the anatomy: a short-axis view of the heart inside the chest wall. The exact path, from the same samples, does not. Use the exact path when exact invertibility and the discrete transform's own rules matter, and the ring-consistent route to reconstruct measured data. The dataset does not record the direction of its first line relative to the patient, so the image may be a rotated or mirrored view of the scanner's own.

## Your own data

Data from any other source goes through `radial_lines_to_polar` directly: the lines as `(readout, line[, batch])`, each line's angle, and the readout index of the center of k-space. Here an off-center Gaussian's analytic k-space is acquired along 32 lines in an interleaved order, even lines first; the conversion places every line by its angle, so the order does not change the result. A golden-angle acquisition is rejected with `NonUniformAnglesError`.

In [ ]:
# A Gaussian's 2-D Fourier transform, acquired along 32 lines of 64 samples
n_lines, n_readout, center = 32, 64, 32
frequencies = (np.arange(n_readout) - center) * (2 * np.pi / n_readout)


def acquire(angles: np.ndarray) -> np.ndarray:
    # The transform of exp(-|x - (4, -2)|^2 / 9) along lines at these angles
    kx = frequencies[:, np.newaxis] * np.cos(angles)
    ky = frequencies[:, np.newaxis] * np.sin(angles)
    envelope = 9 * np.pi * np.exp(-9 * (kx**2 + ky**2) / 4)
    return envelope * np.exp(-1j * (4 * kx - 2 * ky))


# Sequential angles, and the same angles interleaved: even lines first
sequential = np.arange(n_lines) * (np.pi / n_lines)
interleaved = np.concatenate((sequential[0::2], sequential[1::2]))
first = radial_lines_to_polar(
    lines=acquire(angles=sequential), angles=sequential, center_index=center
)
second = radial_lines_to_polar(
    lines=acquire(angles=interleaved), angles=interleaved, center_index=center
)
same = np.array_equal(a1=first.values, a2=second.values)
print(f"{first.values.shape}, same spokes: {same}")

# Golden-angle lines have no uniform polar layout
golden = np.arange(n_lines) * np.deg2rad(111.246117975)
try:
    radial_lines_to_polar(lines=acquire(angles=golden), angles=golden, center_index=center)
except NonUniformAnglesError as error:
    print(f"rejected: {error}")

## Saving, and the terminal interface

`RadialKSpace.save` writes the values and the radius to a NumPy `.npz` file, and `RadialKSpace.load` reads them back. The directory below is temporary; change `output_directory` to the directory you want the file saved in.

The same selection is available without writing code: `python -m pypft.acquisition <file>` opens a terminal interface with the file and its layout, an **Inspect** button reading the array and listing its selectable axes (for a `.npz` or `.mat` file it also lists the file's arrays, and inspecting again reads the one chosen), one field per axis taking an index or `all`, and an output file. **Convert** reads the selection and saves it; errors are shown on screen. It needs no graphical toolkit, only the terminal.

In [ ]:
# Change this to the directory you want the file saved in
output_directory = Path(tempfile.mkdtemp(prefix="pypft_kspace_"))

# Save the batch of coils, and read it back
output_path = output_directory / "P01_coils.npz"
batch.save(path=output_path)
loaded = RadialKSpace.load(path=output_path)
print(f"saved to {output_path}")
print(f"loaded {loaded.values.shape}, radius {loaded.radius:.4f}")

## References

1. <a id="ref-1"></a> H. El-Rewaidy, "Replication Data for: Multi-Domain Convolutional Neural Network (MD-CNN) For Radial Reconstruction of Dynamic Cardiac MRI," Harvard Dataverse, V2, 2020. [doi:10.7910/DVN/CI3WB6](https://doi.org/10.7910/DVN/CI3WB6)
2. <a id="ref-2"></a> H. El-Rewaidy, A. S. Fahmy, F. Pashakhanloo, X. Cai, S. Kucukseymen, I. Csecs, U. Neisius, H. Haji-Valizadeh, B. Menze, and R. Nezafat, "Multi-Domain Convolutional Neural Network (MD-CNN) for Radial Reconstruction of Dynamic Cardiac MRI," *Magnetic Resonance in Medicine*, 85(3):1195-1208, 2021. [doi:10.1002/mrm.28485](https://doi.org/10.1002/mrm.28485)